# Which other books use the same words?

You have read a handful of books. Not many -- five, say -- but they are *yours*: a reader's history, the first thing we know about a person that popularity never used. Unit 2 recommended the same crowded head to everyone and we saw the price. Now we ask the first **reader-dependent** question of the book: **given the books you have read, which *other* books use the same words and themes?**

This is **lexical** (content) retrieval. Every catalog book carries a **keyword document** -- a bag of real-word tokens describing it -- and a reader's profile is simply the keywords of the books they have read. Retrieve the books whose words match. In this unit we build that path from scratch -- **bag-of-words**, then **TF-IDF with cosine similarity**, then **Okapi BM25** -- reveal the `bookrec` implementation, and score it honestly on the Unit 1 scoreboard: it beats the random floor by more than 10x and beats Unit 2's popularity path, which makes it a strong *content candidate source*. It is also, as we will be careful to say, comparable to simple content matching, and it will sit below the collaborative and latent paths still to come.

## Load the catalog, the interactions, and the keyword text

The Unit 1/2 substrate is unchanged: `bookrec.generated_dir()` locates the seeded data slice (run the recsys generators first if it complains), `load_catalog` gives the `dict[int, Book]`, and the interaction log is the gzip-ed CSV with one row per reader-item event. New in this unit is a **separate artifact**, `keywords.csv.gz`: one row per book, `item_id,keywords`, where `keywords` is a space-joined bag of tokens. `bookrec.load_keywords` returns `{item_id: "token token token ..."}`.

In [ ]:
import json
from collections import Counter

import bookrec
import numpy as np
import pandas as pd

data_dir = bookrec.generated_dir()
catalog = bookrec.load_catalog(data_dir / "catalog.csv.gz")
keywords = bookrec.load_keywords(data_dir / "keywords.csv.gz")
interactions_path = data_dir / "interactions.csv.gz"
interactions = pd.read_csv(interactions_path)
catalog_ids = sorted(catalog)

assert set(keywords) == set(catalog), "every catalog book has exactly one keyword document"
len(catalog), len(keywords), len(interactions)

## 1. Bag-of-words: a book is a bag of tokens

A **bag-of-words** model throws away word order and keeps only *which* tokens appear and *how many times*. A book becomes a multiset of tokens; a corpus becomes a table with one row per document and one column per distinct token. Three quantities carry the whole model:

- **vocabulary** -- the set of distinct tokens across the corpus, indexed $t = 0, \dots, V-1$;
- **term frequency** $\mathrm{tf}_{d,t}$ -- how many times token $t$ occurs in document $d$ (real repetition: a keyword used three times counts three);
- **document frequency** $\mathrm{df}_t$ -- how many *documents* contain token $t$ at least once.

```text
  "warehouse hull paddle warehouse breeze ... warehouse"   (one book's keyword bag)
          |
          |  tokenize on whitespace, keep repeats
          v
  Counter({warehouse: 3, hull: 2, paddle: 2, breeze: 1, ...})      <- tf for this document
          |
          |  stack every book's counts into one table
          v
  tf matrix  (n_docs x n_terms),  df = how many rows are non-zero in each column
```

Tokenization here is deliberately trivial -- `str.split()` on whitespace -- because the keyword text is already a clean bag. Real text needs lower-casing, punctuation stripping and stemming before this step; none of that changes the mathematics below.

### One book's token document

Pick a book and look at its bag. The genres from the catalog are the *human* summary; the keyword bag is the *machine-readable* one we will retrieve on.

In [ ]:
def tokenize(text):
    return text.split()


example_id = 1
example_book = catalog[example_id]
example_tokens = tokenize(keywords[example_id])

print(f"{example_book.title}  genres={example_book.fields['genres']}")
print(f"{len(example_tokens)} tokens, {len(set(example_tokens))} distinct")
print("tf (most repeated first):", Counter(example_tokens).most_common(6))

### Vocabulary and document frequency by hand

Tokenize every book once, collect the vocabulary, and count $\mathrm{df}_t$ by adding each document's *set* of tokens to a `Counter`. Two facts about this corpus will matter for the rest of the unit: documents vary a lot in **length** (shortest to longest is about a 3x range), and tokens **repeat only lightly** inside a document (the typical book's most-repeated token appears just twice).

In [ ]:
tokens_of = {item_id: tokenize(keywords[item_id]) for item_id in catalog_ids}

vocab = sorted({token for tokens in tokens_of.values() for token in tokens})
col_of = {token: col for col, token in enumerate(vocab)}

df_counter = Counter()
for tokens in tokens_of.values():
    df_counter.update(set(tokens))

doc_lengths = np.array([len(tokens_of[item_id]) for item_id in catalog_ids])
max_tf = np.array([max(Counter(tokens_of[item_id]).values()) for item_id in catalog_ids])

print(f"documents: {len(catalog_ids)}   vocabulary: {len(vocab)} distinct tokens")
print(f"document length: min {doc_lengths.min()}, median {int(np.median(doc_lengths))}, "
      f"mean {doc_lengths.mean():.1f}, max {doc_lengths.max()}")
print(f"most-repeated token per document: median tf {int(np.median(max_tf))}, max tf {max_tf.max()}")
print()
print("most widespread tokens (df = number of books containing the token):")
for token, df in df_counter.most_common(5):
    print(f"  {token:>10} df={df:4d}  ({df / len(catalog_ids):.1%} of books)")
print("least widespread:")
for token, df in df_counter.most_common()[-3:]:
    print(f"  {token:>10} df={df:4d}  ({df / len(catalog_ids):.1%} of books)")

### The document-term matrix

Now the same information as a dense numpy array $\mathbf{T}$ of shape `(n_docs, n_terms)` with $\mathbf{T}_{d,t} = \mathrm{tf}_{d,t}$. Dense is fine at 2000 x 608; a real corpus would use a sparse matrix, and nothing below depends on the storage. Two sanity checks tie the matrix back to the by-hand counts: each row sums to the document's length, and the column-wise count of non-zeros is exactly $\mathrm{df}$.

In [ ]:
n_docs, n_terms = len(catalog_ids), len(vocab)
tf = np.zeros((n_docs, n_terms))
for row, item_id in enumerate(catalog_ids):
    for token in tokens_of[item_id]:
        tf[row, col_of[token]] += 1.0

df = (tf > 0).sum(axis=0)

assert np.array_equal(tf.sum(axis=1), doc_lengths)
assert all(df[col_of[token]] == count for token, count in df_counter.items())
print(f"tf matrix {tf.shape}; non-zero entries {int((tf > 0).sum())} "
      f"of {tf.size} ({(tf > 0).mean():.1%} dense)")

## 2. TF-IDF and cosine similarity

Raw counts have a problem: a token that appears in a hundred books (`sculptor`, above) says almost nothing about *which* book you want, yet it contributes as much as a token that appears in forty. **Inverse document frequency** fixes that by weighting each token by how rare it is across the corpus. We use the *probabilistic* ("plus-one") form that BM25 also uses, so one $\mathrm{idf}$ serves both scorers:

$$\mathrm{idf}_t = \log\!\left(1 + \frac{N - \mathrm{df}_t + 0.5}{\mathrm{df}_t + 0.5}\right)$$

where $N$ is the number of documents. Read it at its limits: a token in every document ($\mathrm{df}_t = N$) gets $\log(1 + 0.5/(N+0.5)) \approx 0$; a token in a single document gets about $\log(N)$. The $+1$ inside the log keeps every weight strictly positive, and the two $0.5$s are a smoothing that keeps the formula finite at $\mathrm{df}_t = 0$ and $\mathrm{df}_t = N$. The textbook $\log(N/\mathrm{df}_t)$ orders tokens the same way; this form is simply the one the package ships.

The **TF-IDF weight** of token $t$ in document $d$ is the product $w_{d,t} = \mathrm{tf}_{d,t} \cdot \mathrm{idf}_t$, and a document becomes the vector $\mathbf{w}_d \in \mathbb{R}^V$.

**Cosine similarity** compares a query vector $\mathbf{q}$ to a document vector by the angle between them, not their lengths:

$$\cos(\mathbf{q}, \mathbf{w}_d) = \frac{\mathbf{q} \cdot \mathbf{w}_d}{\lVert \mathbf{q} \rVert \, \lVert \mathbf{w}_d \rVert}$$

Dividing by the norms is what stops a long document from winning merely by having more words: a 40-token book and a 12-token book that talk about the same things point in the same direction. If we L2-normalize every document row up front, the cosine against every book is a single matrix-vector product.

In [ ]:
idf = np.log(1.0 + (n_docs - df + 0.5) / (df + 0.5))

for token in ("sculptor", "warehouse"):
    col = col_of[token]
    print(f"{token:>10}: df={int(df[col]):4d}  idf={idf[col]:.3f}")
print(f"idf range across the vocabulary: {idf.min():.3f} .. {idf.max():.3f}")

weighted = tf * idf                                   # broadcast idf across every row
row_norms = np.linalg.norm(weighted, axis=1, keepdims=True)
tfidf_by_hand = weighted / row_norms                  # every row now has unit length

assert np.allclose(np.linalg.norm(tfidf_by_hand, axis=1), 1.0)
tfidf_by_hand.shape

### A reader's keyword profile

A reader's query is the keyword tokens of the books they have read -- their **positive train** interactions, exactly the `seen` set the scoreboard uses. Concatenate those bags into one longer bag, build its TF-IDF vector the same way, and the cosine against every catalog row ranks the books. The reader's own books are excluded (a re-read is not a recommendation), which is what `rank(exclude=seen)` is for.

Reader 3 has read five books. Their combined profile is about 140 tokens and already repeats a few words several times -- repetition *across* a reader's books is the signal we are chasing.

In [ ]:
train_positive = interactions[(interactions["split"] == "train") & (interactions["label"] == 1)]
seen_by = train_positive.groupby("reader_id")["item_id"].apply(set).to_dict()

reader_id = 3
seen = seen_by[reader_id]
profile_tokens = [token for item_id in sorted(seen) for token in tokens_of[item_id]]

print(f"reader {reader_id} has read {sorted(seen)}")
print(f"profile: {len(profile_tokens)} tokens; most repeated: {Counter(profile_tokens).most_common(4)}")

query = np.zeros(n_terms)
for token in profile_tokens:
    query[col_of[token]] += 1.0
query *= idf                                          # the same tf * idf weighting

cosine_by_hand = (tfidf_by_hand @ query) / np.linalg.norm(query)   # rows already unit length

candidates = [
    bookrec.Candidate(item_id, float(cosine_by_hand[row]), "tfidf-by-hand")
    for row, item_id in enumerate(catalog_ids)
]
top5 = bookrec.rank(candidates, n=5, exclude=seen)
print()
print(f"{'item':>6} {'cosine':>7} {'len':>4}  shared tokens with the profile")
profile_set = set(profile_tokens)
for c in top5:
    shared = sorted(profile_set & set(tokens_of[c.item_id]))
    print(f"{c.item_id:>6} {c.score:>7.3f} {len(tokens_of[c.item_id]):>4}  {' '.join(shared[:6])} ...")

### Reveal `BM25Index`, `tfidf_matrix` and `cosine_similarity`

The package packages the by-hand pieces. `bookrec.BM25Index(documents)` tokenizes the `{item_id: text}` corpus and stores exactly the objects we built -- `vocab`, `tf`, `df`, `idf`, `doc_len`, `avgdl` -- with rows in sorted `item_id` order. `bookrec.tfidf_matrix(index)` returns the L2-normalized TF-IDF rows and `bookrec.cosine_similarity(query_vector, matrix)` the cosine of one weighted query against every row. The library's vocabulary is numbered in first-seen order rather than alphabetically, so we compare through its column map rather than position by position; the numbers must agree to floating-point precision.

In [ ]:
index = bookrec.BM25Index(keywords)
assert index.item_ids == catalog_ids
assert set(index.vocab) == set(vocab)

# the library's columns, re-ordered into our alphabetical column order
permutation = np.array([index.vocab[token] for token in vocab])
assert np.array_equal(index.tf[:, permutation], tf)
assert np.allclose(index.idf[permutation], idf)

tfidf_lib = bookrec.tfidf_matrix(index)
assert np.allclose(tfidf_lib[:, permutation], tfidf_by_hand)

query_lib = np.zeros(len(index.vocab))
for token in profile_tokens:
    query_lib[index.vocab[token]] += 1.0
query_lib *= index.idf
cosine_lib = bookrec.cosine_similarity(query_lib, tfidf_lib)
assert np.allclose(cosine_lib, cosine_by_hand)
print("by-hand TF-IDF cosine == bookrec.tfidf_matrix + cosine_similarity")

## 3. BM25: what it adds over TF-IDF cosine, and when it matters

TF-IDF cosine treats term frequency as *linear* -- a token that occurs 6 times contributes 6 times what a single occurrence does -- and it handles document length only indirectly, by normalizing the whole vector. **Okapi BM25** is the search engine's refinement of the same two ideas. For a query $q$ (a multiset of tokens) and a document $d$:

$$\mathrm{BM25}(q, d) = \sum_{t \in q} \mathrm{idf}_t \cdot \frac{\mathrm{tf}_{d,t}\,(k_1 + 1)}{\mathrm{tf}_{d,t} + k_1\left(1 - b + b\,\dfrac{|d|}{\mathrm{avgdl}}\right)}$$

where $|d|$ is the document's length and $\mathrm{avgdl}$ the corpus average. Two parameters, two ideas:

- **$k_1$ -- term-frequency saturation.** The fraction $\frac{\mathrm{tf}(k_1+1)}{\mathrm{tf} + k_1}$ (at average length) rises from 0 at $\mathrm{tf}=0$ and *flattens toward $k_1 + 1$*: the second occurrence of a word is worth less than the first, the tenth almost nothing. A document that says `warehouse` forty times is not forty times more about warehouses. Small $k_1$ saturates fast (almost binary presence); large $k_1$ approaches linear tf.
- **$b$ -- length normalization.** The denominator grows with $|d| / \mathrm{avgdl}$ scaled by $b \in [0, 1]$. At $b = 0$ length is ignored: a long document gets *full* credit for every matched term, and since a long bag simply contains more tokens, long documents collect more matches and float to the top. At $b = 1$ a document twice the average length needs its terms to appear about twice as often to score the same.

```text
   contribution of one matched term, as a function of tf (avg-length doc)
   tf-idf cosine (numerator):  tf * idf        -- straight line, never flattens
   bm25:  idf * tf(k1+1)/(tf+k1)               -- rises fast, bends, flattens toward (k1+1) idf

   score
     ^        . . . . . . . . . . . . . . . . .   <- bm25 ceiling (k1+1) * idf
     |      .   ---------------------------------   bm25 (k1 = 1.2)
     |    .  /                 _____------
     |   . /         ____-----
     |  ./  ____-----                  tf-idf keeps climbing  ---> /
     | /----
     +------------------------------------------------>  tf
       0    1    2    3    4    5    6
```

Note what BM25 is *not*: it is not a cosine. The query is a bag too, and a query token repeated $n$ times simply adds its contribution $n$ times (the shipped `BM25Index.score` multiplies by the query count). And BM25 reuses the same $\mathrm{idf}_t$ from section 2 unchanged; what changes is how `tf` and document length enter.

### $k_1$ saturation, mechanically -- and honestly

First the mechanism on a tiny fixture: one matched term, one document of average length, `tf` running 0 to 6. Then the honest question: *how often does saturation actually bite on this corpus?* We saw that the typical book's most-repeated token appears twice, so most `(document, term)` cells sit at tf = 1 where every $k_1$ curve agrees. Saturation is a real and important idea -- it is why BM25 dominates in web search over documents with thousands of words -- but on short, lightly-repeating keyword bags it is a **small** effect, and we will say so when we score.

In [ ]:
tf_grid = np.arange(0, 7)
print("saturated tf term  tf*(k1+1)/(tf+k1)  at average document length (b term = 1)")
print(f"{'tf':>4}  " + "  ".join(f"k1={k1:<4}" for k1 in (0.5, 1.2, 5.0)) + "   linear (tf-idf)")
for t in tf_grid:
    row = "  ".join(f"{t * (k1 + 1.0) / (t + k1):6.3f} " for k1 in (0.5, 1.2, 5.0))
    print(f"{t:>4}  {row}   {t:6.1f}")

nonzero = tf[tf > 0]
print()
print(f"on this corpus: {np.mean(nonzero == 1):.1%} of non-zero (document, term) cells have tf = 1, "
      f"{np.mean(nonzero >= 3):.1%} have tf >= 3  -> saturation rarely has anything to saturate")

### $b$ length normalization on the real, variable-length documents

Document length is where this corpus *does* give BM25 something to do: bags run from 12 to 40 tokens around an average of about 26. We implement BM25 by hand against the `tf`, `df`, `idf` and `doc_lengths` already in memory, check it against `BM25Index.score`, and then rank reader 3's unseen books at $b = 0$ (no length normalization) and $b = 1$ (full). Watch the lengths of the top-10 move.

In [ ]:
avgdl = doc_lengths.mean()


def bm25_scores(query_tokens, k1=1.2, b=0.75):
    # BM25 of every document against a query token bag, from the by-hand tf / idf arrays.
    query_tf = Counter(token for token in query_tokens if token in col_of)
    length_term = k1 * (1.0 - b + b * doc_lengths / avgdl)       # shape (n_docs,)
    scores = np.zeros(n_docs)
    for token, q_count in query_tf.items():
        col = tf[:, col_of[token]]
        scores += q_count * idf[col_of[token]] * (col * (k1 + 1.0)) / (col + length_term)
    return scores


assert np.allclose(bm25_scores(profile_tokens), index.score(profile_tokens))
assert np.allclose(bm25_scores(profile_tokens, k1=2.0, b=0.0),
                   bookrec.BM25Index(keywords, k1=2.0, b=0.0).score(profile_tokens))
print(f"by-hand BM25 == BM25Index.score   (avgdl = {avgdl:.1f} tokens)")
print()

for b in (0.0, 0.75, 1.0):
    scores = bm25_scores(profile_tokens, b=b)
    pool = [bookrec.Candidate(i, float(scores[row]), "bm25") for row, i in enumerate(catalog_ids)]
    top10 = bookrec.rank(pool, n=10, exclude=seen)
    lengths = [len(tokens_of[c.item_id]) for c in top10]
    print(f"b={b:<4} reader {reader_id} top-10 lengths {lengths}  mean {np.mean(lengths):.1f}")

One reader is an anecdote; the length bias is a property of the scorer, so measure it over *every* reader with a training history: the mean document length of each reader's top-10, averaged across readers, at each $b$.

In [ ]:
print(f"corpus average document length: {avgdl:.1f}")
for b in (0.0, 0.75, 1.0):
    top_lengths = []
    for r_id, r_seen in seen_by.items():
        r_tokens = [token for item_id in r_seen for token in tokens_of[item_id]]
        scores = bm25_scores(r_tokens, b=b)
        order = np.argsort(-scores, kind="stable")
        top_rows = [row for row in order if catalog_ids[row] not in r_seen][:10]
        top_lengths.append(doc_lengths[top_rows].mean())
    print(f"b={b:<4} mean length of recommended books, over {len(seen_by)} readers: "
          f"{np.mean(top_lengths):.1f}")

At $b = 0$ the recommended books average about **36** tokens against a corpus mean of **26** -- the ranker is quietly rewarding *long* bags, which have more chances to contain any given profile word. Raising $b$ pulls the recommended length down toward the corpus mean (about **33** at the default $0.75$, about **32** at $b = 1$). That is the length-normalization effect, visible on real documents: **$b$ matters whenever document lengths vary and length is not itself evidence of relevance.** A longer keyword bag here does not mean a better match; it just means more words.

Whether the un-biased ranking is also the more *accurate* one is an empirical question for the scoreboard -- we will check, and the answer is "slightly".

### Reveal `LexicalRetrievalPath`, register it, and score it

`bookrec.LexicalRetrievalPath(keywords, k1=1.2, b=0.75)` takes the **keyword corpus at construction** -- mirroring `RandomRetrievalPath(item_ids)` -- so that `fit(interactions, catalog=None)` keeps the exact `RetrievalPath` signature and the path is fully substitutable in the registry. `fit` ignores the interaction log (a content path indexes the catalog text, not who read what) and builds the `BM25Index`; `retrieve(query, context, k)` reads the reader's `context["seen"]`, concatenates those books' keyword bags into the query, scores every document with BM25, drops the seen books, and returns the top `k` as `Candidate`s carrying raw BM25 scores (calibrated later, at blend time, by `path.calibrate`). A reader with no history gets `[]` -- the honest answer from a path that depends on history, and the reason popularity stays as the cold-start fallback.

In [ ]:
rows = interactions.to_dict("records")

lexical = bookrec.LexicalRetrievalPath(keywords).fit(rows, catalog=catalog_ids)
popularity = bookrec.PopularityRetrievalPath().fit(rows, catalog=catalog_ids)
random_floor = bookrec.RandomRetrievalPath(catalog_ids, seed=0)

registry = bookrec.PathRegistry()
for path in (random_floor, popularity, lexical):
    registry.register(path)

shipped = lexical.retrieve(query=reader_id, context={"seen": seen}, k=5)
by_hand = bookrec.rank(
    [bookrec.Candidate(i, float(s), "bm25") for i, s in zip(catalog_ids, bm25_scores(profile_tokens))],
    n=5, exclude=seen,
)
assert [c.item_id for c in shipped] == [c.item_id for c in by_hand]

print(f"reader {reader_id}, top-5 from the shipped path:")
for c in shipped:
    print(f"  item {c.item_id:>5}  score {c.score:.3f}  genres {catalog[c.item_id].fields['genres']}")
print()
print("no history -> no profile -> no recommendations:",
      lexical.retrieve(query=999_999, context={"seen": set()}, k=5))

Four of reader 3's five books carry the `poetry` genre, and every one of the five recommendations does too -- yet the path never read the genre column. The words found the theme. That is content retrieval working as intended, and also a first hint at its ceiling: matching words is a proxy for matching genres, so we should expect this path to perform *like* simple content matching, not far above it.

Now the scoreboard, under the Unit 1 rules: hit-rate@10 over positive `val` rows, each reader's train-positive items removed from both recommendations and relevance, `cold_readers` from `cold_partitions.json` excluded, random floor seeded at 0.

In [ ]:
cold = json.loads((data_dir / "cold_partitions.json").read_text())
cold_readers = set(cold["cold_readers"])


def score(path):
    return bookrec.run_validation_scoreboard(
        path, interactions_path, catalog_ids=catalog_ids, k=10, cold_readers=cold_readers
    )


random_board = score(random_floor)
pop_board = score(popularity)
lexical_board = score(lexical)

print(f"readers scored: {lexical_board.readers}")
print(f"{'path':>12} {'hit@10':>8} {'recall@10':>10} {'vs floor':>9}")
for label, board in (("random", random_board), ("popularity", pop_board), ("lexical", lexical_board)):
    print(f"{label:>12} {board.hit_rate_at_k:>8.4f} {board.recall_at_k:>10.4f} "
          f"{board.hit_rate_at_k / random_board.hit_rate_at_k:>8.1f}x")
print(f"\nlexical / popularity: {lexical_board.hit_rate_at_k / pop_board.hit_rate_at_k:.2f}x")

### Read the scoreboard honestly

Lexical BM25 scores about **0.16** hit@10 against a random floor of about **0.012** -- comfortably **more than 10x the floor** -- and about **1.5x** Unit 2's popularity path (about **0.11**), on the same 500 readers. The first reader-dependent path wins, and it wins by *reading the catalog*, not the crowd: it will happily recommend a book no one else has read, as long as its words match yours. That is the opposite failure mode from popularity's head bias, and it is why lexical retrieval earns a permanent place as a **content candidate source** in the blend.

Keep the win in proportion. The keyword text was built so that a book's words correlate with its genres and themes, so matching words is in the same family as matching *genres* -- lexical retrieval is **comparable to simple content matching**, a solid floor-beater rather than a breakthrough. It also cannot see the signal the collaborative and latent paths of Units 4 and 5 will use: *other readers like you*. Expect those paths to sit above it.

### Does BM25 beat TF-IDF cosine on the scoreboard?

We should check rather than assume. Wrap the by-hand TF-IDF cosine as a path (same protocol, same `seen` handling), score it, and sweep BM25's two parameters.

In [ ]:
class TfidfCosinePath(bookrec.LexicalRetrievalPath):
    # The by-hand TF-IDF cosine ranker, dressed as a RetrievalPath for the scoreboard.

    def fit(self, interactions, catalog=None):
        super().fit(interactions, catalog)
        self._matrix = bookrec.tfidf_matrix(self._index)
        return self

    def retrieve(self, query, context, k):
        idx = self._index
        reader_seen = set(context["seen"])
        q = np.zeros(len(idx.vocab))
        for item_id in reader_seen:
            for token in tokenize(self._keywords[item_id]):
                q[idx.vocab[token]] += 1.0
        q *= idx.idf
        sims = bookrec.cosine_similarity(q, self._matrix)
        pool = [bookrec.Candidate(int(i), float(sims[row]), self.name)
                for row, i in enumerate(idx.item_ids) if i not in reader_seen]
        return self._finish(pool, k)


tfidf_board = score(TfidfCosinePath(keywords, name="tfidf-cosine").fit(rows, catalog=catalog_ids))
print(f"tf-idf cosine hit@10: {tfidf_board.hit_rate_at_k:.4f}")
print(f"bm25 (k1=1.2, b=0.75): {lexical_board.hit_rate_at_k:.4f}")

In [ ]:
print("bm25 parameter sweep, hit@10:")
for k1, b in ((1.2, 0.0), (1.2, 1.0), (0.5, 0.75), (5.0, 0.75)):
    board = score(bookrec.LexicalRetrievalPath(keywords, k1=k1, b=b, name=f"bm25-{k1}-{b}")
                  .fit(rows, catalog=catalog_ids))
    print(f"  k1={k1:<4} b={b:<5} {board.hit_rate_at_k:.4f}")

**No, not here -- and that is the honest lesson.** TF-IDF cosine and BM25 land within a few thousandths of each other (both about **0.16**), and every BM25 setting in the sweep sits in the same narrow band (about 0.15 to 0.17). Both of BM25's refinements are real, and both are *small on this corpus* for reasons we have already measured: saturation has almost nothing to saturate when nearly every matched term has tf = 1, and length normalization moves the recommended length visibly (section 3) but only nudges accuracy, with $b = 1$ scoring a touch higher than $b = 0$ and no setting clearly best. On a corpus of long, repetitive, highly variable documents -- web pages, papers, product reviews -- BM25's edge over TF-IDF cosine is large and consistent, which is why it is the default ranking function in every open-source search engine. On 26-token keyword bags, the two are peers.

So the package ships BM25 as `LexicalRetrievalPath` because it is the standard, parameterized, production form of lexical scoring, not because it won a race here. If a later unit swaps in TF-IDF cosine for speed, nothing on the scoreboard should move.

## Where this goes

Three ideas, one path, and an honest number:

- **Bag-of-words** turns text into counts: vocabulary, `tf`, `df`. Everything downstream is linear algebra on that table.
- **TF-IDF + cosine** weights rare words up and compares *directions*, so document length stops being a free advantage. A reader's profile is just the bag of what they have read.
- **BM25** adds saturating tf ($k_1$) and explicit length normalization ($b$). On this corpus $b$ is demonstrable and $k_1$ is mechanical; neither beats cosine on hit@10, and we said so.

`LexicalRetrievalPath` is the first path in the registry whose output depends on the reader. It beats the floor by more than 10x and beats popularity, it can surface books nobody else has read, and it fails gracefully to `[]` with no history -- which is exactly where popularity steps back in. What it cannot do is learn from *other readers*: two books with no words in common that the same people keep reading together are invisible to it. That signal -- collaborative filtering -- is **Unit 4**, and the blend of popularity, lexical and collaborative candidates is where the recommender starts to feel like one.